<a href="https://colab.research.google.com/github/abdulbaset2016/ai-ethics-operationalization-framework/blob/main/ethics_ai_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/abdulbaset2016/worldwide_AI-ethicss.git
#!pip install -q bertopic sentence-transformers umap-learn hdbscan


Cloning into 'worldwide_AI-ethicss'...
remote: Enumerating objects: 473, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 473 (delta 2), reused 0 (delta 0), pack-reused 465 (from 1)
Receiving objects: 100% (473/473), 167.22 MiB | 43.01 MiB/s, done.
Resolving deltas: 100% (256/256), done.


In [ ]:

import os
import re
import json
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import chi2_contingency, kendalltau, spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.linear_model import LinearRegression
import plotly.express as px
import plotly.graph_objects as go

# ================================================================
# GLOBAL SETTINGS
# ================================================================

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ================================================================
# MAIN FRAMEWORK
# ================================================================

class EthicalOperationalizationFramework:

    # ------------------------------------------------------------
    # Core ethical principles (expanded with sensitivity constructs)
    # ------------------------------------------------------------

    CORE_PRINCIPLES = [
        "ACCOUNTABILITY", "TRANSPARENCY", "PRIVACY", "FAIRNESS",
        "RELIABILITY", "HUMAN_DIGNITY", "FREEDOM_AND_AUTONOMY",
        "DIVERSITY_AND_INCLUSION", "JUSTICE_AND_EQUITY", "BENEFICENCE",
        "COOPERATION", "SUSTAINABILITY", "HUMAN_FORMATION",
        "LABOR_RIGHTS", "HUMAN_CENTEREDNESS", "INTELLECTUAL_PROPERTY",
        "TRUTHFULNESS", "TRUST", "ROBUSTNESS", "DEPENDABILITY", "SAFETY",
        "DIGNITY", "SOLIDARITY"
    ]

    # ------------------------------------------------------------
    # RAI Dimension Columns (based on ACTUAL dataset structure)
    # ------------------------------------------------------------

    RAI_DIMENSION_COLUMNS = [
        "Connections to Harm",
        "Measurement Properties",
        "Algorithmic System Characteristics",
        "Publication Metadata",
        "Target Output"
    ]

    RAI_DIMENSION_LABELS = {
        "Connections to Harm": "Connections to Harm",
        "Measurement Properties": "Measurement Properties",
        "Algorithmic System Characteristics": "Algorithmic System Characteristics",
        "Publication Metadata": "Publication Metadata",
        "Target Output": "Target Output"
    }

    # ------------------------------------------------------------
    # Principle Aliases for guidelines normalization
    # ------------------------------------------------------------

    PRINCIPLE_ALIASES = {
        "JUSTICE": "JUSTICE_AND_EQUITY",
        "EQUITY": "JUSTICE_AND_EQUITY",
        "JUSTICE_EQUITY": "JUSTICE_AND_EQUITY",
        "JUSTICE_AND_EQUITY": "JUSTICE_AND_EQUITY",
        "HUMAN_DIGNITY": "HUMAN_DIGNITY",
        "FREEDOM": "FREEDOM_AND_AUTONOMY",
        "AUTONOMY": "FREEDOM_AND_AUTONOMY",
        "FREEDOM_AND_AUTONOMY": "FREEDOM_AND_AUTONOMY",
        "DIVERSITY": "DIVERSITY_AND_INCLUSION",
        "INCLUSION": "DIVERSITY_AND_INCLUSION",
        "DIVERSITY_AND_INCLUSION": "DIVERSITY_AND_INCLUSION",
        "HUMAN_CENTERED": "HUMAN_CENTEREDNESS",
        "HUMAN_CENTEREDNESS": "HUMAN_CENTEREDNESS",
        "INTELLECTUAL_PROPERTY": "INTELLECTUAL_PROPERTY",
        "TRUTH": "TRUTHFULNESS",
        "TRUTHFULNESS": "TRUTHFULNESS",
        "ACCOUNTABILITY": "ACCOUNTABILITY",
        "TRANSPARENCY": "TRANSPARENCY",
        "PRIVACY": "PRIVACY",
        "FAIRNESS": "FAIRNESS",
        "RELIABILITY": "RELIABILITY",
        "BENEFICENCE": "BENEFICENCE",
        "COOPERATION": "COOPERATION",
        "SUSTAINABILITY": "SUSTAINABILITY",
        "HUMAN_FORMATION": "HUMAN_FORMATION",
        "LABOR_RIGHTS": "LABOR_RIGHTS",
        "TRUST": "TRUST",
        "ROBUSTNESS": "ROBUSTNESS",
        "DEPENDABILITY": "DEPENDABILITY",
        "SAFETY": "SAFETY",
        "DIGNITY": "DIGNITY",
        "SOLIDARITY": "SOLIDARITY",
        "RESPONSIBILITY": "ACCOUNTABILITY",
        "NON-MALEFICENCE": "BENEFICENCE",
        "NON_MALEFICENCE": "BENEFICENCE",
        "FREEDOM": "FREEDOM_AND_AUTONOMY"
    }

    # ------------------------------------------------------------
    # RAI Canonical Mapping (for RAI dataset only)
    # ------------------------------------------------------------

    RAI_CANONICAL_MAPPING = {
        "fairness": "FAIRNESS",
        "justice": "FAIRNESS",
        "equity": "FAIRNESS",
        "transparency": "TRANSPARENCY",
        "privacy": "PRIVACY",
        "reliability": "RELIABILITY",
        "accountability": "ACCOUNTABILITY",
        "responsibility": "ACCOUNTABILITY",
        "freedom and autonomy": "FREEDOM_AND_AUTONOMY",
        "autonomy": "FREEDOM_AND_AUTONOMY",
        "diversity and inclusion": "DIVERSITY_AND_INCLUSION",
        "beneficence": "BENEFICENCE",
        "non-maleficence": "BENEFICENCE",
        "non_maleficence": "BENEFICENCE",
        "cooperation": "COOPERATION",
        "human formation": "HUMAN_FORMATION",
        "labor rights": "LABOR_RIGHTS",
        "human centeredness": "HUMAN_CENTEREDNESS",
        "sustainability": "SUSTAINABILITY",
        "intellectual property": "INTELLECTUAL_PROPERTY",
        "truthfulness": "TRUTHFULNESS",
        "trust": "TRUST",
        "dignity": "DIGNITY",
        "solidarity": "SOLIDARITY",
        "robustness": "ROBUSTNESS",
        "dependability": "DEPENDABILITY",
        "safety": "SAFETY"
    }

    # ------------------------------------------------------------
    # EOGI Thresholds
    # ------------------------------------------------------------

    EOGI_THRESHOLDS = {
        "High_Gap": 0.50,
        "Moderate_Gap": 0.25,
        "Low_Gap": 0.00
    }

    # ------------------------------------------------------------
    # Reliability Sensitivity Mapping
    # ------------------------------------------------------------

    RELIABILITY_SENSITIVITY_MAPPING = {
        "strict": {"RELIABILITY"},
        "trust_inclusive": {"RELIABILITY", "TRUST"},
        "broad": {"RELIABILITY", "TRUST", "ROBUSTNESS", "DEPENDABILITY", "SAFETY"}
    }

    # ============================================================
    # INITIALIZATION
    # ============================================================

    def __init__(
        self,
        guidelines_path: str,
        rai_data_path: Optional[str] = None,
        seed: int = GLOBAL_SEED,
        n_permutations: int = 5000,
        n_bootstrap: int = 1000
    ):

        self.guidelines_path = guidelines_path
        self.rai_data_path = rai_data_path
        self.seed = seed
        self.n_permutations = n_permutations
        self.n_bootstrap = n_bootstrap
        self.rng = np.random.default_rng(seed)

        # Data containers
        self.df = None
        self.rai_df = None
        self.rai_raw = None
        self.principles = []
        self.rai_column_map = {}

        # Analysis results
        self.P = None
        self.regional_analysis = None
        self.regional_statistics = None
        self.temporal = None
        self.temporal_sen = None
        self.network = None
        self.clusters = None
        self.cluster_jaccard = None

        # RAI results
        self.M = None
        self.rai_profile = None
        self.operationalization_index = None
        self.rai_counts_by_principle = None

        # EOGI results
        self.eogi_baseline = None
        self.eogi_v2 = None
        self.eogi_bootstrap = None
        self.eogi_permutation = None
        self.weight_sensitivity = None
        self.eogi_loo = None
        self.eogi_validation = None
        self.reliability_sensitivity = None

        self.paradox = None

        # --------------------------------------------------------
        # Load guideline dataset
        # --------------------------------------------------------

        self.df = self.load_guidelines()
        if self.df is None:
            raise RuntimeError("Failed to load guidelines dataset.")
        if len(self.df) == 0:
            raise RuntimeError("Guidelines dataset is empty.")

        self.principles = self.get_principle_columns()
        if len(self.principles) < 2:
            raise RuntimeError("At least two ethical principle columns are required.")
        logger.info(f"Detected {len(self.principles)} ethical principles.")

        # --------------------------------------------------------
        # Main analyses
        # --------------------------------------------------------

        self.P = self.calculate_principle_prevalence()
        self.regional_analysis = self.calculate_regional()
        self.regional_statistics = self.calculate_regional_statistics()
        self.temporal = self.calculate_weighted_temporal()
        self.temporal_sen = self.calculate_sen_slope()
        self.network = self.calculate_network()
        self.clusters = self.calculate_clusters()
        self.cluster_jaccard = self.calculate_jaccard_clustering()

        # --------------------------------------------------------
        # RAI analysis
        # --------------------------------------------------------

        self.M, self.rai_counts_by_principle = self.load_rai_measures()
        if self.M is not None and len(self.M) > 0:
            self.rai_profile = self.build_rai_operationalization_profile()
            self.operationalization_index = self.calculate_operationalization_index()
            self.eogi_baseline = self.calculate_baseline_eogi()
            self.eogi_v2 = self.calculate_eogi_v2()
            self.weight_sensitivity = self.eogi_weight_sensitivity()
            self.eogi_permutation = self.eogi_permutation_test()
            self.eogi_loo = self.leave_one_out_sensitivity()
            self.eogi_validation = self.validate_eogi()
            self.reliability_sensitivity = self.calculate_reliability_sensitivity()

        self.paradox = self.identify_reliability_gap()

    # ============================================================
    # GENERAL UTILITIES
    # ============================================================

    @staticmethod
    def clean_name(name: str) -> str:
        """Clean and standardize principle names."""
        if pd.isna(name):
            return ""
        text = str(name).strip().upper()
        replacements = {
            "&": "AND", "/": "_", "-": "_", " ": "_",
            "(": "", ")": "", ",": "", ".": "",
            "'": "", '"': ""
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        text = re.sub(r"_+", "_", text)
        return text.strip("_")

    @classmethod
    def normalize_principle(cls, value: str) -> str:
        """Normalize principle name using alias mapping."""
        cleaned = cls.clean_name(value)
        return cls.PRINCIPLE_ALIASES.get(cleaned, cleaned)

    @staticmethod
    def safe_minmax(series: pd.Series) -> pd.Series:
        """Safe min-max normalization with zero division protection."""
        series = pd.to_numeric(series, errors="coerce").fillna(0)
        mn, mx = series.min(), series.max()
        if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn <= 1e-12:
            return pd.Series(0.0, index=series.index)
        return (series - mn) / (mx - mn)

    @staticmethod
    def safe_spearman(x, y, min_obs: int = 3) -> Tuple[float, float]:
        """Safe Spearman correlation with handling of edge cases."""
        x = pd.to_numeric(pd.Series(x), errors="coerce")
        y = pd.to_numeric(pd.Series(y), errors="coerce")
        valid = x.notna() & y.notna()
        x_clean = x[valid].values
        y_clean = y[valid].values

        if len(x_clean) < min_obs:
            return np.nan, np.nan
        if np.std(x_clean) == 0 or np.std(y_clean) == 0:
            return np.nan, np.nan

        rho, p = spearmanr(x_clean, y_clean)
        return float(rho), float(p)

    @staticmethod
    def parse_entry_points(value) -> List[str]:
        """
        Parse the multi-label 'Entry Points' field from RAI dataset.
        Handles comma, semicolon, pipe, and newline separators.
        """
        if pd.isna(value):
            return []

        text = str(value).strip()
        if not text:
            return []

        # Split on comma, semicolon, pipe, or newline
        parts = re.split(r"[,;|\n]+", text)

        cleaned = []
        for p in parts:
            p = str(p).strip().lower()
            p = re.sub(r"\s+", " ", p)  # Normalize whitespace
            if p:
                cleaned.append(p)

        # Remove duplicates while preserving order
        return list(dict.fromkeys(cleaned))

    @classmethod
    def map_rai_principles(cls, principles: List[str]) -> List[str]:
        """Map RAI entry points to canonical principles."""
        mapped = []
        for p in principles:
            canonical = cls.RAI_CANONICAL_MAPPING.get(p)
            if canonical is not None:
                mapped.append(canonical)
        return list(dict.fromkeys(mapped))

    @staticmethod
    def split_multivalue(value) -> List[str]:
        """Split multi-value fields for dimension values."""
        if pd.isna(value):
            return []
        if isinstance(value, (list, tuple, set)):
            return [str(v).strip() for v in value if str(v).strip()]
        text = str(value).strip()
        if not text:
            return []
        parts = re.split(r"\s*(?:,|;|\||\n)\s*", text)
        return [x.strip() for x in parts if x.strip()]

    @staticmethod
    def shannon_diversity(values: List[str]) -> float:
        """Calculate Shannon diversity index."""
        if not values:
            return 0.0
        counts = pd.Series(values).value_counts().values
        probabilities = counts / counts.sum()
        return float(-np.sum(probabilities * np.log(probabilities + 1e-12)))

    @staticmethod
    def normalized_shannon_diversity(values: List[str], n_categories: int = None) -> float:
        """
        Calculate normalized Shannon diversity.
        H_norm = H / ln(K) where K is number of possible categories.
        Returns 0 <= H_norm <= 1.
        """
        if not values:
            return 0.0

        counts = pd.Series(values).value_counts()
        k = len(counts) if n_categories is None else n_categories

        if k <= 1:
            return 0.0

        probabilities = counts.values / counts.values.sum()
        H = -np.sum(probabilities * np.log(probabilities + 1e-12))
        return float(H / np.log(k))

    @staticmethod
    def wilson_ci(successes: int, n: int, z: float = 1.96) -> Tuple[float, float]:
        """Wilson confidence interval for binomial proportion."""
        if n <= 0:
            return np.nan, np.nan
        p = successes / n
        denominator = 1 + z**2 / n
        centre = p + z**2 / (2 * n)
        margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
        return (centre - margin) / denominator, (centre + margin) / denominator

    @staticmethod
    def sen_slope(x: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        """Calculate Sen's slope estimator for trend analysis."""
        n = len(x)
        if n < 2:
            return {"slope": np.nan, "intercept": np.nan, "p_value": np.nan}

        slopes = []
        for i in range(n):
            for j in range(i + 1, n):
                if x[j] != x[i]:
                    slopes.append((y[j] - y[i]) / (x[j] - x[i]))

        if not slopes:
            return {"slope": np.nan, "intercept": np.nan, "p_value": np.nan}

        slopes = np.array(slopes)
        slope = np.median(slopes)
        intercepts = y - slope * x
        intercept = np.median(intercepts)
        tau, p_value = kendalltau(x, y)

        return {
            "slope": float(slope),
            "intercept": float(intercept),
            "p_value": float(p_value),
            "kendall_tau": float(tau),
            "n_pairs": len(slopes)
        }

    # ============================================================
    # GUIDELINES LOADING
    # ============================================================

    def load_guidelines(self) -> Optional[pd.DataFrame]:
        """Load guidelines dataset from JSON or Parquet."""
        json_path = os.path.join(self.guidelines_path, "data_processed.json")
        parquet_path = os.path.join(self.guidelines_path, "data_raw.parquet")

        try:
            if os.path.exists(json_path):
                logger.info(f"Loading guidelines JSON: {json_path}")
                with open(json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                return self.process_guidelines_dataframe(pd.DataFrame(data))

            elif os.path.exists(parquet_path):
                logger.info(f"Loading guidelines Parquet: {parquet_path}")
                return self.process_guidelines_dataframe(pd.read_parquet(parquet_path))

            else:
                logger.error("No guideline dataset found.")
                return None

        except Exception as e:
            logger.exception(f"Error loading guidelines: {e}")
            return None

    def process_guidelines_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process and clean the guidelines dataframe."""
        processed = df.copy()

        # Map common metadata columns
        col_map = {
            "world_region": "region",
            "institution_type": "sector",
            "year_of_publication": "year"
        }
        for old, new in col_map.items():
            if old in processed.columns:
                processed[new] = processed[old]

        # Document ID
        if "document_id" not in processed.columns:
            processed["document_id"] = np.arange(len(processed))

        # Year
        processed["year"] = pd.to_numeric(
            processed.get("year", np.nan),
            errors="coerce"
        )

        # Metadata columns
        defaults = {"region": "Global", "sector": "Mixed", "country": "Various"}
        for col in ["region", "sector", "country"]:
            if col in processed.columns:
                processed[col] = processed[col].fillna("Unknown").astype(str).str.strip()
            else:
                processed[col] = defaults.get(col, "Unknown")

        # Extract principles from dictionary
        if "principles" in processed.columns:
            all_principles = set()
            for value in processed["principles"]:
                if isinstance(value, dict):
                    for key in value.keys():
                        all_principles.add(self.normalize_principle(key))

            for principle in all_principles:
                if principle:
                    processed[principle] = processed["principles"].apply(
                        lambda x: int(
                            isinstance(x, dict) and
                            any(self.normalize_principle(k) == principle and bool(v)
                                for k, v in x.items())
                        )
                    )

        # Normalize existing principle columns
        for col in list(processed.columns):
            cleaned = self.normalize_principle(col)
            if cleaned in self.CORE_PRINCIPLES:
                if cleaned != col:
                    processed[cleaned] = processed[col]
                processed[cleaned] = pd.to_numeric(
                    processed[cleaned], errors="coerce"
                ).fillna(0).gt(0).astype(int)

        # Ensure all core principles exist
        for principle in self.CORE_PRINCIPLES:
            if principle not in processed.columns:
                processed[principle] = 0

        return processed

    def get_principle_columns(self) -> List[str]:
        """Get list of principles present in the data."""
        return [
            p for p in self.CORE_PRINCIPLES
            if p in self.df.columns and self.df[p].sum() > 0
        ]

    # ============================================================
    # PREVALENCE & REGIONAL ANALYSIS
    # ============================================================

    def calculate_principle_prevalence(self) -> pd.DataFrame:
        """Calculate principle prevalence with Wilson CI."""
        n = len(self.df)
        records = []

        for principle in self.principles:
            successes = int(self.df[principle].sum())
            ci_low, ci_high = self.wilson_ci(successes, n)

            records.append({
                "principle": principle,
                "prevalence": successes / n,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "documents": successes,
                "total_documents": n
            })

        return pd.DataFrame(records).sort_values(
            "prevalence", ascending=False
        ).reset_index(drop=True)

    def calculate_regional(self) -> pd.DataFrame:
        """Calculate regional prevalence for each principle."""
        records = []

        for principle in self.principles:
            for region in sorted(self.df["region"].dropna().unique()):
                subset = self.df[self.df["region"] == region]
                if len(subset) == 0:
                    continue

                records.append({
                    "principle": principle,
                    "region": region,
                    "prevalence": subset[principle].mean(),
                    "count": int(subset[principle].sum()),
                    "total": len(subset)
                })

        return pd.DataFrame(records)

    def calculate_regional_statistics(self) -> Dict:
        """Calculate χ², Cramér's V, and standardized residuals with FDR correction."""
        results = {}

        for principle in self.principles:
            contingency = pd.crosstab(self.df["region"], self.df[principle])

            if contingency.shape[0] < 2 or contingency.shape[1] != 2:
                continue

            try:
                chi2, p_value, dof, expected = chi2_contingency(contingency)
                n = contingency.values.sum()
                min_dim = min(contingency.shape)
                cramer_v = np.sqrt(chi2 / (n * (min_dim - 1))) if min_dim > 1 else np.nan

                expected_df = pd.DataFrame(
                    expected,
                    index=contingency.index,
                    columns=contingency.columns
                )
                residuals = (contingency - expected_df) / np.sqrt(expected_df + 1e-12)
                contribution = ((contingency - expected_df) ** 2) / (expected_df + 1e-12)

                results[principle] = {
                    "chi2": float(chi2),
                    "p_value": float(p_value),
                    "dof": int(dof),
                    "cramer_v": float(cramer_v) if np.isfinite(cramer_v) else np.nan,
                    "contingency": contingency,
                    "expected": expected_df,
                    "standardized_residuals": residuals,
                    "contribution": contribution,
                    "n": int(n)
                }

            except Exception as e:
                logger.warning(f"Regional test failed for {principle}: {e}")

        # FDR correction
        if results:
            names = list(results.keys())
            p_values = [results[p]["p_value"] for p in names]
            reject, adjusted, _, _ = multipletests(
                p_values, alpha=0.05, method="fdr_bh"
            )

            for i, principle in enumerate(names):
                results[principle]["p_adjusted"] = float(adjusted[i])
                results[principle]["significant_fdr"] = bool(reject[i])

        return results

    # ============================================================
    # TEMPORAL ANALYSIS
    # ============================================================

    def calculate_weighted_temporal(self) -> Dict:
        """Calculate weighted linear regression trends (supplementary)."""
        trends = {}
        temporal_df = self.df.dropna(subset=["year"]).copy()

        if temporal_df.empty:
            return trends

        for principle in self.principles:
            yearly = temporal_df.groupby("year")[principle].agg(
                prevalence="mean",
                n_docs="count"
            ).reset_index().sort_values("year")

            if len(yearly) < 3:
                continue

            X = yearly["year"].values.astype(float)
            y = yearly["prevalence"].values.astype(float)
            weights = yearly["n_docs"].values.astype(float) / yearly["n_docs"].sum()

            x_bar, y_bar = np.sum(weights * X), np.sum(weights * y)
            xc, yc = X - x_bar, y - y_bar
            denominator = np.sum(weights * xc ** 2)

            if denominator <= 0:
                continue

            slope = np.sum(weights * xc * yc) / denominator
            intercept = y_bar - slope * x_bar
            y_pred = slope * X + intercept
            residuals = y - y_pred
            df_error = len(X) - 2

            if df_error > 0:
                mse = np.sum(weights * residuals ** 2) / df_error
                se_slope = np.sqrt(mse / denominator)
                if se_slope > 0:
                    t_stat = slope / se_slope
                    p_reg = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_error))
                else:
                    p_reg = np.nan
            else:
                se_slope, p_reg = np.nan, np.nan

            weighted_sse = np.sum(weights * residuals ** 2)
            weighted_tss = np.sum(weights * yc ** 2)
            weighted_r2 = 1 - weighted_sse / weighted_tss if weighted_tss > 0 else np.nan

            trends[principle] = {
                "slope": float(slope),
                "annual_change_pp": float(slope * 100),
                "std_error": float(se_slope) if np.isfinite(se_slope) else np.nan,
                "p_value_regression": float(p_reg) if np.isfinite(p_reg) else np.nan,
                "weighted_r2": float(weighted_r2) if np.isfinite(weighted_r2) else np.nan,
                "ci_lower": float(slope - 1.96 * se_slope) if np.isfinite(se_slope) else np.nan,
                "ci_upper": float(slope + 1.96 * se_slope) if np.isfinite(se_slope) else np.nan,
                "n_years": len(yearly),
                "total_docs": int(yearly["n_docs"].sum()),
                "yearly_data": yearly
            }

        return trends
    def calculate_sen_slope(self) -> Dict:
        """Calculate Sen's slope estimator for each principle (primary trend)."""
        results = {}
        temporal_df = self.df.dropna(subset=["year"]).copy()

        if temporal_df.empty:
            return results

        for principle in self.principles:
            yearly = temporal_df.groupby("year")[principle].agg(
                prevalence="mean",
                n_docs="count"
            ).reset_index().sort_values("year")

            if len(yearly) < 3:
                continue

            sen_result = self.sen_slope(
                yearly["year"].values.astype(float),
                yearly["prevalence"].values.astype(float)
            )
            sen_result["principle"] = principle
            sen_result["n_years"] = len(yearly)
            sen_result["total_docs"] = int(yearly["n_docs"].sum())
            sen_result["yearly_data"] = yearly
            results[principle] = sen_result

        # ============================================================
        # FDR Correction for Multiple Comparisons
        # ============================================================
        if results:
            principles = list(results.keys())
            p_values = [results[p]["p_value"] for p in principles]

            reject, p_adjusted, _, _ = multipletests(
                p_values,
                alpha=0.05,
                method='fdr_bh'
            )

            for i, principle in enumerate(principles):
                results[principle]["p_adjusted"] = float(p_adjusted[i])
                results[principle]["significant_fdr"] = bool(reject[i])

        return results
    # ============================================================
    # NETWORK ANALYSIS (Signed Phi)
    # ============================================================

    @staticmethod
    def signed_phi(x: np.ndarray, y: np.ndarray) -> float:
        """Calculate signed phi coefficient for binary variables."""
        table = pd.crosstab(pd.Series(x), pd.Series(y))

        if table.shape != (2, 2):
            return np.nan

        try:
            a, b, c, d = table.iloc[0, 0], table.iloc[0, 1], table.iloc[1, 0], table.iloc[1, 1]
            denominator = np.sqrt((a + b) * (c + d) * (a + c) * (b + d))
            if denominator == 0:
                return np.nan
            return (a * d - b * c) / denominator

        except Exception:
            return np.nan

    def calculate_phi_matrix(self) -> pd.DataFrame:
        """Calculate full phi matrix for all principles."""
        n = len(self.principles)
        matrix = pd.DataFrame(
            np.eye(n),
            index=self.principles,
            columns=self.principles
        )

        for i in range(n):
            for j in range(i + 1, n):
                phi = self.signed_phi(
                    self.df[self.principles[i]].values,
                    self.df[self.principles[j]].values
                )
                matrix.iloc[i, j] = phi
                matrix.iloc[j, i] = phi

        return matrix

    def bootstrap_phi_ci(self, p1: str, p2: str, n_bootstrap: int = 1000) -> Dict:
        """Calculate bootstrap confidence intervals for phi."""
        x = self.df[p1].values
        y = self.df[p2].values
        n = len(x)

        phi_values = []
        for _ in range(n_bootstrap):
            idx = self.rng.choice(n, size=n, replace=True)
            phi = self.signed_phi(x[idx], y[idx])
            if np.isfinite(phi):
                phi_values.append(phi)

        if not phi_values:
            return {"ci_lower": np.nan, "ci_upper": np.nan, "mean": np.nan}

        phi_values = np.array(phi_values)
        return {
            "mean": float(np.mean(phi_values)),
            "ci_lower": float(np.percentile(phi_values, 2.5)),
            "ci_upper": float(np.percentile(phi_values, 97.5)),
            "std": float(np.std(phi_values))
        }

    def calculate_network(self) -> Dict:
        """Calculate network structure with permutation testing and bootstrap CI."""
        phi_matrix = self.calculate_phi_matrix()
        relations = []

        for i, j in combinations(range(len(self.principles)), 2):
            p1, p2 = self.principles[i], self.principles[j]
            x, y = self.df[p1].values, self.df[p2].values
            phi_obs = phi_matrix.loc[p1, p2]

            if not np.isfinite(phi_obs):
                continue

            # Bootstrap CI
            boot_ci = self.bootstrap_phi_ci(p1, p2, min(self.n_bootstrap, 500))

            # Permutation test
            perm_vals = []
            for _ in range(self.n_permutations):
                phi_perm = self.signed_phi(self.rng.permutation(x), y)
                if np.isfinite(phi_perm):
                    perm_vals.append(phi_perm)

            if not perm_vals:
                continue

            perm_vals = np.array(perm_vals)
            p_value = (np.sum(np.abs(perm_vals) >= abs(phi_obs)) + 1) / (len(perm_vals) + 1)

            relations.append({
                "principle_a": p1,
                "principle_b": p2,
                "phi": float(phi_obs),
                "phi_ci_lower": boot_ci["ci_lower"],
                "phi_ci_upper": boot_ci["ci_upper"],
                "p_value": float(p_value),
                "n_permutations": len(perm_vals),
                "n_bootstrap": self.n_bootstrap
            })

        # FDR correction
        if relations:
            p_values = [r["p_value"] for r in relations]
            reject, adjusted, _, _ = multipletests(
                p_values, alpha=0.05, method="fdr_bh"
            )

            for i, rel in enumerate(relations):
                rel["p_adjusted"] = float(adjusted[i])
                rel["significant"] = bool(reject[i])
                abs_phi = abs(rel["phi"])

                if abs_phi >= 0.5:
                    rel["strength"] = "strong"
                elif abs_phi >= 0.3:
                    rel["strength"] = "moderate"
                elif abs_phi >= 0.1:
                    rel["strength"] = "weak"
                else:
                    rel["strength"] = "negligible"

                rel["direction"] = "positive" if rel["phi"] > 0 else "negative"

        validated = [r for r in relations if r.get("significant", False)]
        possible_edges = len(self.principles) * (len(self.principles) - 1) / 2
        density = len(validated) / possible_edges if possible_edges > 0 else 0

        # Centrality
        centrality = {}
        for principle in self.principles:
            edges = [
                r for r in validated
                if r["principle_a"] == principle or r["principle_b"] == principle
            ]
            centrality[principle] = {
                "degree": len(edges),
                "weighted_degree": sum(abs(r["phi"]) for r in edges)
            }

        return {
            "phi_matrix": phi_matrix,
            "all_relationships": relations,
            "validated_relationships": validated,
            "network_density": float(density),
            "centrality": pd.DataFrame(centrality).T.sort_values(
                "weighted_degree", ascending=False
            )
        }

    # ============================================================
    # CLUSTERING ANALYSIS
    # ============================================================

    def find_optimal_clusters(self, max_k: int = 10) -> Dict:
        """Find optimal number of clusters using multiple metrics."""
        X = self.df[self.principles].values
        n = len(X)

        if n < 4:
            return {"optimal_k": 2, "scores": pd.DataFrame()}

        max_k = min(max_k, n - 1)
        records = []

        for k in range(2, max_k + 1):
            try:
                model = KMeans(n_clusters=k, random_state=self.seed, n_init=20)
                labels = model.fit_predict(X)

                if len(np.unique(labels)) < 2:
                    continue

                records.append({
                    "k": k,
                    "silhouette": silhouette_score(X, labels),
                    "calinski_harabasz": calinski_harabasz_score(X, labels),
                    "davies_bouldin": davies_bouldin_score(X, labels),
                    "inertia": model.inertia_
                })

            except Exception as e:
                logger.warning(f"Clustering k={k} failed: {e}")

        score_df = pd.DataFrame(records)

        if score_df.empty:
            return {"optimal_k": 2, "scores": score_df}

        # Normalize metrics
        score_df["sil_norm"] = self.safe_minmax(score_df["silhouette"])
        score_df["cal_norm"] = self.safe_minmax(score_df["calinski_harabasz"])

        db = score_df["davies_bouldin"]
        if db.max() - db.min() <= 1e-12:
            score_df["dav_norm"] = 1.0
        else:
            score_df["dav_norm"] = 1 - ((db - db.min()) / (db.max() - db.min()))

        score_df["composite"] = (score_df["sil_norm"] + score_df["cal_norm"] + score_df["dav_norm"]) / 3
        best_idx = score_df["composite"].idxmax()

        return {
            "optimal_k": int(score_df.loc[best_idx, "k"]),
            "scores": score_df
        }

    def calculate_bootstrap_stability(self, n_clusters: int, n_bootstrap: int = None) -> Dict:
        """Calculate bootstrap stability using Adjusted Rand Index."""
        if n_bootstrap is None:
            n_bootstrap = self.n_bootstrap

        X = self.df[self.principles].values
        n = len(X)

        original_labels = KMeans(
            n_clusters=n_clusters, random_state=self.seed, n_init=20
        ).fit_predict(X)

        ari_scores = []

        for b in range(min(n_bootstrap, 500)):
            indices = self.rng.choice(n, size=n, replace=True)
            boot_model = KMeans(
                n_clusters=n_clusters,
                random_state=self.seed + b + 1,
                n_init=20
            )
            boot_model.fit(X[indices])

            distances = ((X[:, None, :] - boot_model.cluster_centers_[None, :, :]) ** 2).sum(axis=2)
            reassigned = np.argmin(distances, axis=1)

            ari = adjusted_rand_score(original_labels, reassigned)
            ari_scores.append(ari)

        ari_scores = np.array(ari_scores)

        return {
            "mean_ari": float(np.mean(ari_scores)),
            "std_ari": float(np.std(ari_scores)),
            "median_ari": float(np.median(ari_scores)),
            "ci_lower": float(np.percentile(ari_scores, 2.5)),
            "ci_upper": float(np.percentile(ari_scores, 97.5)),
            "stable": bool(np.mean(ari_scores) >= 0.70),
            "ari_values": ari_scores
        }

    def calculate_jaccard_clustering(self, n_clusters: int = None) -> Dict:
        """Calculate clustering using Jaccard/Hamming distance for robustness."""
        X = self.df[self.principles].values
        n = len(X)

        if n_clusters is None:
            n_clusters = self.clusters["n_clusters"] if self.clusters else 2

        n_clusters = max(2, min(n_clusters, n - 1))

        # Hamming distance
        hamming_dist = pairwise_distances(X, metric="hamming")
        linkage_matrix = linkage(hamming_dist, method="ward")
        h_labels = fcluster(linkage_matrix, t=n_clusters, criterion="maxclust") - 1

        # KMeans on binary data
        kmeans_hamming = KMeans(
            n_clusters=n_clusters,
            random_state=self.seed,
            n_init=20
        )
        km_labels = kmeans_hamming.fit_predict(X)

        if self.clusters is not None:
            km_labels_original = self.clusters["clusters"]
            ari_h = adjusted_rand_score(km_labels_original, h_labels)
            ari_km = adjusted_rand_score(km_labels_original, km_labels)
        else:
            ari_h = ari_km = np.nan

        return {
            "n_clusters": n_clusters,
            "hierarchical_labels": h_labels,
            "kmeans_hamming_labels": km_labels,
            "ari_with_kmeans_hierarchical": float(ari_h) if np.isfinite(ari_h) else np.nan,
            "ari_with_kmeans_hamming": float(ari_km) if np.isfinite(ari_km) else np.nan,
            "linkage_matrix": linkage_matrix
        }

    def calculate_clusters(self, n_clusters: int = None) -> Dict:
        """Main clustering analysis using KMeans."""
        X = self.df[self.principles].values

        selection = self.find_optimal_clusters()
        if n_clusters is None:
            n_clusters = selection["optimal_k"]

        n_clusters = max(2, min(int(n_clusters), len(X) - 1))

        # PCA for visualization
        pca_components = min(3, X.shape[1], X.shape[0])
        pca = PCA(n_components=pca_components, random_state=self.seed)
        X_pca = pca.fit_transform(X)

        # KMeans
        kmeans = KMeans(n_clusters=n_clusters, random_state=self.seed, n_init=20)
        clusters = kmeans.fit_predict(X)
        self.df["cluster"] = clusters

        stability = self.calculate_bootstrap_stability(n_clusters)

        return {
            "clusters": clusters,
            "profiles": self.df.groupby("cluster")[self.principles].mean(),
            "regional": self.df.groupby("cluster")["region"].value_counts(
                normalize=True
            ).unstack(fill_value=0),
            "institutional": self.df.groupby("cluster")["sector"].value_counts(
                normalize=True
            ).unstack(fill_value=0),
            "pca_variance": pca.explained_variance_ratio_,
            "pca_coordinates": X_pca,
            "inertia": float(kmeans.inertia_),
            "centroids": kmeans.cluster_centers_,
            "n_clusters": n_clusters,
            "stability": stability,
            "selection": selection
        }

    # ============================================================
    # RAI DATA LOADING - FULLY RESTRUCTURED
    # ============================================================

    def load_rai_measures(self) -> Tuple[Optional[Dict], Optional[Dict]]:
        """
        Load and prepare RAI measures dataset.
        Handles the actual dataset structure with 'Entry Points' as principle column.
        Returns:
            - counts_by_principle: number of measures per canonical principle
            - counts_by_entry: raw entry point counts (for reference)
        """
        if not self.rai_data_path or not os.path.exists(self.rai_data_path):
            logger.warning("No RAI dataset supplied or found.")
            return None, None

        try:
            path = self.rai_data_path.lower()

            if path.endswith(".csv"):
                rai_df = pd.read_csv(self.rai_data_path)
            elif path.endswith(".xlsx"):
                rai_df = pd.read_excel(self.rai_data_path)
            else:
                raise ValueError("Unsupported RAI file format.")

            self.rai_raw = rai_df.copy()
            logger.info(f"RAI dataset loaded: {len(rai_df)} rows")
            logger.info(f"Columns: {list(rai_df.columns)}")

            # ----------------------------------------------------
            # Step 1: Remove header-description row if present
            # ----------------------------------------------------
            if "Target Output" in rai_df.columns:
                rai_df = rai_df[
                    rai_df["Target Output"].astype(str).str.strip().str.lower()
                    != "measure"
                ].copy()

            # ----------------------------------------------------
            # Step 2: Create unique measure ID
            # ----------------------------------------------------
            rai_df = rai_df.reset_index(drop=True)
            rai_df["measure_id"] = np.arange(len(rai_df))

            # ----------------------------------------------------
            # Step 3: Verify Entry Points exists
            # ----------------------------------------------------
            if "Entry Points" not in rai_df.columns:
                raise ValueError(
                    f"RAI dataset must contain 'Entry Points' column. "
                    f"Available columns: {list(rai_df.columns)}"
                )

            # ----------------------------------------------------
            # Step 4: Parse multi-label principles
            # ----------------------------------------------------
            rai_df["entry_points_list"] = rai_df["Entry Points"].apply(
                self.parse_entry_points
            )

            # Remove rows without valid principles
            rai_df = rai_df[
                rai_df["entry_points_list"].apply(len) > 0
            ].copy()

            # ----------------------------------------------------
            # Step 5: Map to canonical principles
            # ----------------------------------------------------
            rai_df["canonical_principles"] = rai_df["entry_points_list"].apply(
                self.map_rai_principles
            )

            # Remove rows without canonical principles
            rai_df = rai_df[
                rai_df["canonical_principles"].apply(len) > 0
            ].copy()

            self.rai_df = rai_df

            # ----------------------------------------------------
            # Step 6: Count measures by canonical principle
            # ----------------------------------------------------
            counts = Counter()
            for principles in rai_df["canonical_principles"]:
                for principle in set(principles):
                    counts[principle] += 1

            counts_by_principle = dict(counts)

            # Also count raw entry points for diagnostics
            raw_counts = Counter()
            for entry_points in rai_df["entry_points_list"]:
                for ep in set(entry_points):
                    raw_counts[ep] += 1

            # ----------------------------------------------------
            # Step 7: Map RAI dimension columns (actual column names)
            # ----------------------------------------------------
            self.rai_column_map = {}

            # Map actual columns to our dimension names
            for dim_name in self.RAI_DIMENSION_COLUMNS:
                if dim_name in rai_df.columns:
                    self.rai_column_map[dim_name] = dim_name
                else:
                    logger.warning(f"Dimension column '{dim_name}' not found in RAI dataset")

            # Add any additional columns that might be useful
            extra_columns = ["Connections to Harm", "Measurement Properties",
                           "Algorithmic System Characteristics", "Publication Metadata"]
            for col in extra_columns:
                if col in rai_df.columns and col not in self.rai_column_map:
                    self.rai_column_map[col] = col

            logger.info(f"RAI dataset prepared: {len(rai_df)} valid rows")
            logger.info(f"Mapped dimensions: {list(self.rai_column_map.keys())}")
            logger.info(f"Total canonical entries: {sum(counts_by_principle.values())}")
            logger.info(f"Unique canonical principles: {len(counts_by_principle)}")
            logger.info(f"Sample counts: {dict(list(counts_by_principle.items())[:10])}")

            # ----------------------------------------------------
            # Step 8: Diagnostics
            # ----------------------------------------------------
            self._diagnose_rai_columns()

            return counts_by_principle, dict(raw_counts)

        except Exception as e:
            logger.exception(f"Error loading RAI data: {e}")
            self.rai_df = None
            return None, None

    def _diagnose_rai_columns(self):
        """Diagnose RAI dataset columns for verification."""
        if self.rai_df is None:
            logger.warning("RAI dataset not loaded")
            return

        logger.info("=== RAI DATASET DIAGNOSTICS ===")
        logger.info(f"Total rows: {len(self.rai_df)}")
        logger.info(f"Columns: {list(self.rai_df.columns)}")

        logger.info("--- Dimension Columns ---")
        for col in self.RAI_DIMENSION_COLUMNS:
            if col in self.rai_df.columns:
                non_null = self.rai_df[col].notna().sum()
                unique = self.rai_df[col].nunique()
                logger.info(f"{col}: {non_null} non-null, {unique} unique")
            else:
                logger.warning(f"{col}: NOT FOUND")

        logger.info("--- Entry Points Sample ---")
        logger.info(str(self.rai_df["Entry Points"].head(5).tolist()))

        logger.info("--- Canonical Principles Sample ---")
        logger.info(str(self.rai_df["canonical_principles"].head(5).tolist()))

        logger.info("--- Counts by Canonical Principle ---")
        if self.M:
            for p, count in sorted(self.M.items(), key=lambda x: -x[1])[:10]:
                logger.info(f"  {p}: {count}")

    def get_rai_principle_counts(self) -> Dict:
        """Get counts of measures by canonical principle."""
        if self.rai_counts_by_principle is not None:
            return self.rai_counts_by_principle
        if self.M is not None:
            return self.M
        return {}

    def get_reliability_measures(self, mode: str = "strict") -> pd.DataFrame:
        """
        Get RAI measures for reliability based on mapping mode.

        Args:
            mode: 'strict', 'trust_inclusive', or 'broad'
        """
        if mode not in self.RELIABILITY_SENSITIVITY_MAPPING:
            raise ValueError(f"Unknown reliability mapping mode: {mode}")

        if self.rai_df is None:
            return pd.DataFrame()

        allowed = self.RELIABILITY_SENSITIVITY_MAPPING[mode]

        mask = self.rai_df["canonical_principles"].apply(
            lambda principles: any(p in allowed for p in principles)
        )

        return self.rai_df.loc[mask].copy()

    # ============================================================
    # RAI PROFILE
    # ============================================================

    def build_rai_operationalization_profile(self) -> Optional[pd.DataFrame]:
        """Build RAI operationalization profile for each principle."""
        if self.rai_df is None:
            return None

        records = []

        for principle in self.principles:
            # Get measures that contain this principle
            mask = self.rai_df["canonical_principles"].apply(
                lambda x: principle in x
            )
            subset = self.rai_df.loc[mask].copy()

            row = {
                "principle": principle,
                "measure_count": len(subset)
            }

            # Count dimensions
            for dim_name in self.RAI_DIMENSION_COLUMNS:
                if dim_name not in self.rai_df.columns:
                    row[f"{dim_name}_unique"] = 0
                    row[f"{dim_name}_values"] = []
                    continue

                values = []
                for value in subset[dim_name]:
                    values.extend(self.split_multivalue(value))

                values = [v.strip() for v in values if v.strip()]
                row[f"{dim_name}_unique"] = len(set(values))
                row[f"{dim_name}_values"] = sorted(set(values))

            records.append(row)

        return pd.DataFrame(records)

    # ============================================================
    # OPERATIONALIZATION INDEX (OI) - FULLY RESTRUCTURED
    # ============================================================

    def calculate_dimension_coverage(
        self,
        principle_measures: pd.DataFrame,
        all_rai: pd.DataFrame
    ) -> float:
        """
        Calculate dimension coverage for a set of measures.
        Coverage = number of unique values in principle measures / number of unique values in all measures.
        """
        coverages = []

        for dim_name in self.RAI_DIMENSION_COLUMNS:
            if dim_name not in all_rai.columns:
                continue

            # Global values
            global_values = set()
            for value in all_rai[dim_name].dropna():
                parts = self.split_multivalue(value)
                global_values.update(p.strip().lower() for p in parts if p.strip())

            # Principle-specific values
            principle_values = set()
            for value in principle_measures[dim_name].dropna():
                parts = self.split_multivalue(value)
                principle_values.update(p.strip().lower() for p in parts if p.strip())

            if len(global_values) == 0:
                continue

            coverage = len(principle_values) / len(global_values)
            coverages.append(min(max(coverage, 0.0), 1.0))

        if not coverages:
            return 0.0

        return float(np.mean(coverages))

    def calculate_dimension_diversity(
        self,
        principle_measures: pd.DataFrame
    ) -> float:
        """
        Calculate normalized Shannon diversity across dimensions for a set of measures.
        """
        diversities = []

        for dim_name in self.RAI_DIMENSION_COLUMNS:
            if dim_name not in principle_measures.columns:
                continue

            values = []
            for value in principle_measures[dim_name].dropna():
                parts = self.split_multivalue(value)
                values.extend(p.strip().lower() for p in parts if p.strip())

            d = self.normalized_shannon_diversity(values)
            diversities.append(d)

        if not diversities:
            return 0.0

        return float(np.mean(diversities))

    def calculate_operationalization_index(self) -> Optional[pd.DataFrame]:
        """
        Calculate Operationalization Index (OI) using Equal Weight (1/3 each).
        Components:
            1. Measure Coverage (log-normalized count)
            2. Dimension Coverage (across RAI dimensions)
            3. Diversity (normalized Shannon entropy)
        """
        if self.rai_df is None:
            return None

        records = []

        for principle in self.principles:
            # Get measures for this principle
            mask = self.rai_df["canonical_principles"].apply(
                lambda x: principle in x
            )
            principle_measures = self.rai_df.loc[mask].copy()

            # Measure count
            measure_count = len(principle_measures)

            # Measure coverage (log-normalized count)
            measure_rate = measure_count / len(self.rai_df) if len(self.rai_df) > 0 else 0

            # Dimension coverage
            dimension_coverage = self.calculate_dimension_coverage(
                principle_measures,
                self.rai_df
            )

            # Diversity
            diversity = self.calculate_dimension_diversity(principle_measures)

            row = {
                "principle": principle,
                "measure_count": measure_count,
                "measure_rate": measure_rate,
                "dimension_coverage": dimension_coverage,
                "diversity": diversity
            }

            records.append(row)

        oi = pd.DataFrame(records)

        # Normalize components
        oi["measure_coverage_norm"] = self.safe_minmax(np.log1p(oi["measure_count"]))
        oi["dimension_coverage_norm"] = self.safe_minmax(oi["dimension_coverage"])
        oi["diversity_norm"] = self.safe_minmax(oi["diversity"])

        # PRIMARY: Equal weight (1/3 each)
        oi["operationalization_index"] = (
            oi["measure_coverage_norm"] +
            oi["dimension_coverage_norm"] +
            oi["diversity_norm"]
        ) / 3

        return oi.sort_values("operationalization_index", ascending=False).reset_index(drop=True)

    # ============================================================
    # EOGI - BASELINE
    # ============================================================

    def calculate_baseline_eogi(self) -> Optional[pd.DataFrame]:
        """Calculate baseline EOGI (simple measure count only)."""
        if self.M is None:
            return None

        eogi = pd.DataFrame([
            {
                "principle": p,
                "prevalence": self.df[p].mean(),
                "rai_measures": self.M.get(p, 0)
            }
            for p in self.principles
        ])

        eogi["prevalence_norm"] = self.safe_minmax(eogi["prevalence"])
        eogi["measures_norm"] = self.safe_minmax(np.log1p(eogi["rai_measures"]))
        eogi["eogi_baseline"] = eogi["prevalence_norm"] - eogi["measures_norm"]
        eogi["gap_category"] = eogi["eogi_baseline"].apply(self.classify_eogi)

        return eogi.sort_values("eogi_baseline", ascending=False).reset_index(drop=True)

    # ============================================================
    # EOGI - V2 (Multidimensional)
    # ============================================================

    def calculate_eogi_v2(self) -> Optional[pd.DataFrame]:
        """
        Calculate EOGI v2: EOGI = P - OI
        Where P = normalized prevalence, OI = operationalization index.
        """
        if self.operationalization_index is None:
            return None

        result = self.P[["principle", "prevalence"]].merge(
            self.operationalization_index[
                [
                    "principle",
                    "operationalization_index",
                    "measure_count",
                    "measure_coverage_norm",
                    "dimension_coverage_norm",
                    "diversity_norm"
                ]
            ],
            on="principle",
            how="left"
        )

        result["prevalence_norm"] = self.safe_minmax(result["prevalence"])
        result["eogi_v2"] = result["prevalence_norm"] - result["operationalization_index"]
        result["absolute_gap"] = result["eogi_v2"].abs()
        result["gap_category"] = result["eogi_v2"].apply(self.classify_eogi)

        return result.sort_values("eogi_v2", ascending=False).reset_index(drop=True)

    @classmethod
    def classify_eogi(cls, x: float) -> str:
        """Classify EOGI using pre-defined thresholds."""
        if x >= cls.EOGI_THRESHOLDS["High_Gap"]:
            return "Critical Operationalization Gap"
        elif x >= cls.EOGI_THRESHOLDS["Moderate_Gap"]:
            return "Significant Operationalization Gap"
        elif x >= cls.EOGI_THRESHOLDS["Low_Gap"]:
            return "Moderate Alignment"
        else:
            return "Strong Operational Alignment"

    # ============================================================
    # EOGI - BOOTSTRAP
    # ============================================================

    def bootstrap_eogi_v2(self, n_bootstrap: int = None) -> Optional[pd.DataFrame]:
        """Bootstrap EOGI v2 confidence intervals."""
        if n_bootstrap is None:
            n_bootstrap = min(self.n_bootstrap, 1000)

        if self.rai_df is None:
            return None

        guideline_n = len(self.df)
        rai_n = len(self.rai_df)

        results = {p: [] for p in self.principles}

        for b in range(n_bootstrap):
            # Bootstrap guidelines
            g_indices = self.rng.choice(guideline_n, size=guideline_n, replace=True)
            guideline_boot = self.df.iloc[g_indices]

            # Bootstrap RAI
            r_indices = self.rng.choice(rai_n, size=rai_n, replace=True)
            rai_boot = self.rai_df.iloc[r_indices]

            # Prevalence
            p_norm = self.safe_minmax(
                pd.Series({p: guideline_boot[p].mean() for p in self.principles})
            )

            # Compute OI for bootstrap sample
            boot_oi_results = []
            for principle in self.principles:
                mask = rai_boot["canonical_principles"].apply(
                    lambda x: principle in x
                )
                principle_measures = rai_boot.loc[mask].copy()

                measure_count = len(principle_measures)
                dimension_coverage = self.calculate_dimension_coverage(
                    principle_measures,
                    rai_boot
                )
                diversity = self.calculate_dimension_diversity(principle_measures)

                boot_oi_results.append({
                    "principle": principle,
                    "measure_count": measure_count,
                    "dimension_coverage": dimension_coverage,
                    "diversity": diversity
                })

            boot_oi = pd.DataFrame(boot_oi_results)
            boot_oi["measure_coverage_norm"] = self.safe_minmax(
                np.log1p(boot_oi["measure_count"])
            )
            boot_oi["dimension_coverage_norm"] = self.safe_minmax(
                boot_oi["dimension_coverage"]
            )
            boot_oi["diversity_norm"] = self.safe_minmax(boot_oi["diversity"])

            boot_oi["oi"] = (
                boot_oi["measure_coverage_norm"] +
                boot_oi["dimension_coverage_norm"] +
                boot_oi["diversity_norm"]
            ) / 3

            # EOGI
            oi_series = boot_oi.set_index("principle")["oi"]
            eogi_boot = p_norm - oi_series

            for principle in self.principles:
                results[principle].append(float(eogi_boot.get(principle, np.nan)))

        # Summarize
        records = []
        for principle in self.principles:
            vals = np.array([v for v in results[principle] if np.isfinite(v)])
            if len(vals) == 0:
                continue
            records.append({
                "principle": principle,
                "eogi_mean": float(np.mean(vals)),
                "eogi_median": float(np.median(vals)),
                "ci_lower": float(np.percentile(vals, 2.5)),
                "ci_upper": float(np.percentile(vals, 97.5)),
                "bootstrap_sd": float(np.std(vals))
            })

        return pd.DataFrame(records)

    # ============================================================
    # EOGI - PERMUTATION TEST
    # ============================================================

    def eogi_permutation_test(self, n_permutations: int = None) -> Optional[Dict]:
        """Global maximum-gap permutation test."""
        if n_permutations is None:
            n_permutations = min(self.n_permutations, 5000)

        if self.eogi_v2 is None or self.rai_df is None:
            return None

        observed = self.eogi_v2.set_index("principle")["eogi_v2"]
        observed_max = observed.max()

        rai_original = self.rai_df["canonical_principles"].values.copy()
        null_max = []

        for _ in range(n_permutations):
            temp = self.rai_df.copy()
            temp["canonical_principles"] = self.rng.permutation(rai_original)

            # Compute OI for permuted data
            perm_oi_results = []
            for principle in self.principles:
                mask = temp["canonical_principles"].apply(
                    lambda x: principle in x
                )
                principle_measures = temp.loc[mask].copy()

                measure_count = len(principle_measures)
                dimension_coverage = self.calculate_dimension_coverage(
                    principle_measures,
                    temp
                )
                diversity = self.calculate_dimension_diversity(principle_measures)

                perm_oi_results.append({
                    "principle": principle,
                    "measure_count": measure_count,
                    "dimension_coverage": dimension_coverage,
                    "diversity": diversity
                })

            perm_oi = pd.DataFrame(perm_oi_results)
            perm_oi["measure_coverage_norm"] = self.safe_minmax(
                np.log1p(perm_oi["measure_count"])
            )
            perm_oi["dimension_coverage_norm"] = self.safe_minmax(
                perm_oi["dimension_coverage"]
            )
            perm_oi["diversity_norm"] = self.safe_minmax(perm_oi["diversity"])

            perm_oi["oi"] = (
                perm_oi["measure_coverage_norm"] +
                perm_oi["dimension_coverage_norm"] +
                perm_oi["diversity_norm"]
            ) / 3

            # Prevalence (fixed)
            p_norm = self.safe_minmax(
                self.P.set_index("principle")["prevalence"]
            )

            oi_series = perm_oi.set_index("principle")["oi"]
            null_eogi = p_norm - oi_series
            null_max.append(float(null_eogi.max()))

        null_max = np.array(null_max)
        p_value = (np.sum(null_max >= observed_max) + 1) / (len(null_max) + 1)

        return {
            "observed_max_gap": float(observed_max),
            "null_mean": float(np.mean(null_max)),
            "null_sd": float(np.std(null_max)),
            "null_ci_lower": float(np.percentile(null_max, 2.5)),
            "null_ci_upper": float(np.percentile(null_max, 97.5)),
            "p_value": float(p_value),
            "significant": bool(p_value < 0.05),
            "n_permutations": len(null_max)
        }

    # ============================================================
    # EOGI - WEIGHT SENSITIVITY
    # ============================================================

    def eogi_weight_sensitivity(self) -> Optional[pd.DataFrame]:
        """Analyze EOGI sensitivity to different weighting schemes."""
        if self.operationalization_index is None:
            return None

        data = self.P[["principle", "prevalence"]].merge(
            self.operationalization_index[
                [
                    "principle",
                    "measure_coverage_norm",
                    "dimension_coverage_norm",
                    "diversity_norm"
                ]
            ],
            on="principle"
        )

        data["p_norm"] = self.safe_minmax(data["prevalence"])

        configs = {
            "Equal_Weight": (1/3, 1/3, 1/3),
            "Coverage_Emphasis": (0.50, 0.30, 0.20),
            "Dimensions_Emphasis": (0.20, 0.60, 0.20),
            "Diversity_Emphasis": (0.20, 0.30, 0.50),
            "Measure_Heavy": (0.60, 0.20, 0.20)
        }

        rows = []
        for name, (w1, w2, w3) in configs.items():
            oi = (
                w1 * data["measure_coverage_norm"] +
                w2 * data["dimension_coverage_norm"] +
                w3 * data["diversity_norm"]
            )
            eogi = data["p_norm"] - oi

            for principle, value in zip(data["principle"], eogi):
                rows.append({
                    "configuration": name,
                    "principle": principle,
                    "eogi": float(value)
                })

        return pd.DataFrame(rows)

    # ============================================================
    # EOGI - LEAVE-ONE-OUT
    # ============================================================

    def leave_one_out_sensitivity(self) -> Optional[Dict]:
        """Leave-One-Out (LOO) sensitivity analysis."""
        if self.eogi_v2 is None:
            return None

        full_eogi = self.eogi_v2.set_index("principle")["eogi_v2"]
        results = {}

        for leave_out in self.principles:
            temp_principles = [p for p in self.principles if p != leave_out]

            # Recalculate prevalence without left-out principle
            temp_df = self.df.copy()
            temp_P = temp_df[temp_principles].mean()
            temp_P_norm = self.safe_minmax(temp_P)

            # Recalculate OI without left-out principle
            oi_df = self.operationalization_index[
                self.operationalization_index["principle"] != leave_out
            ].copy()

            oi_df["measure_coverage_norm_loo"] = self.safe_minmax(
                np.log1p(oi_df["measure_count"])
            )
            oi_df["dimension_coverage_norm_loo"] = self.safe_minmax(
                oi_df["dimension_coverage"]
            )
            oi_df["diversity_norm_loo"] = self.safe_minmax(oi_df["diversity"])

            oi_df["oi_loo"] = (
                oi_df["measure_coverage_norm_loo"] +
                oi_df["dimension_coverage_norm_loo"] +
                oi_df["diversity_norm_loo"]
            ) / 3

            # LOO EOGI
            loo_eogi = {}
            for p in temp_principles:
                loo_eogi[p] = temp_P_norm[p] - oi_df[oi_df["principle"] == p]["oi_loo"].iloc[0]

            loo_series = pd.Series(loo_eogi)

            common_principles = [p for p in temp_principles if p in full_eogi.index]
            if len(common_principles) > 1:
                corr, p_corr = spearmanr(
                    [full_eogi[p] for p in common_principles],
                    [loo_series[p] for p in common_principles]
                )
            else:
                corr, p_corr = np.nan, np.nan

            results[leave_out] = {
                "principles_used": temp_principles,
                "n_principles": len(temp_principles),
                "lo_eogi": loo_series.to_dict(),
                "correlation_with_full": float(corr) if np.isfinite(corr) else np.nan,
                "correlation_p": float(p_corr) if np.isfinite(p_corr) else np.nan
            }

        correlations = [
            r["correlation_with_full"]
            for r in results.values()
            if np.isfinite(r["correlation_with_full"])
        ]

        return {
            "leave_one_out_results": results,
            "mean_loo_correlation": float(np.mean(correlations)) if correlations else np.nan,
            "min_loo_correlation": float(np.min(correlations)) if correlations else np.nan,
            "stable": bool(np.mean(correlations) > 0.90) if correlations else False,
            "n_loops": len(results)
        }

    # ============================================================
    # EOGI - VALIDATION
    # ============================================================

    def validate_eogi(self) -> Optional[Dict]:
        """Validate EOGI: construct validity and discriminant validity."""
        if self.eogi_v2 is None:
            return None

        df = self.eogi_v2.copy()

        # Construct validity: correlations
        validations = {
            "correlation_with_prevalence": self.safe_spearman(df["prevalence"], df["eogi_v2"]),
            "correlation_with_oi": self.safe_spearman(df["operationalization_index"], df["eogi_v2"]),
            "correlation_with_measure_count": self.safe_spearman(df["measure_count"], df["eogi_v2"]),
            "correlation_with_measure_coverage": self.safe_spearman(df["measure_coverage_norm"], df["eogi_v2"]),
            "correlation_with_dimension_coverage": self.safe_spearman(df["dimension_coverage_norm"], df["eogi_v2"]),
            "correlation_with_diversity": self.safe_spearman(df["diversity_norm"], df["eogi_v2"])
        }

        # Variance diagnostics
        variance_diagnostics = {}
        for col in [
            "prevalence", "operationalization_index", "measure_count",
            "measure_coverage_norm", "dimension_coverage_norm", "diversity_norm"
        ]:
            variance_diagnostics[col] = {
                "n_unique": int(df[col].nunique(dropna=True)),
                "variance": float(df[col].var()) if df[col].notna().sum() > 1 else np.nan
            }

        # Discriminant validity: EOGI should not be just measure count
        X = np.log1p(df["measure_count"].values).reshape(-1, 1)
        y = df["eogi_v2"].values

        model = LinearRegression().fit(X, y)
        r2_measure_only = model.score(X, y)

        y_pred = model.predict(X)
        residuals = y - y_pred

        residual_corr_with_dim = self.safe_spearman(
            df["dimension_coverage_norm"].values,
            residuals
        )

        return {
            "construct_validity": {
                k: {
                    "correlation": v[0] if not np.isnan(v[0]) else np.nan,
                    "p_value": v[1] if not np.isnan(v[1]) else np.nan,
                    "status": "computed" if not np.isnan(v[0]) else "undefined_zero_variance"
                }
                for k, v in validations.items()
            },
            "variance_diagnostics": variance_diagnostics,
            "discriminant_validity": {
                "r2_measure_only": float(r2_measure_only),
                "residual_correlation_with_dimension": {
                    "correlation": residual_corr_with_dim[0] if not np.isnan(residual_corr_with_dim[0]) else np.nan,
                    "p_value": residual_corr_with_dim[1] if not np.isnan(residual_corr_with_dim[1]) else np.nan
                }
            }
        }

    # ============================================================
    # RELIABILITY SENSITIVITY
    # ============================================================
    def calculate_reliability_sensitivity(self) -> Optional[pd.DataFrame]:

        if self.rai_df is None or self.eogi_v2 is None:
            return None

        # Get reliability prevalence
        reliability_prevalence = self.df["RELIABILITY"].mean()
        p_series = pd.Series({"RELIABILITY": reliability_prevalence})
        p_norm = self.safe_minmax(p_series)["RELIABILITY"]

        results = []

        # Get max measure count for normalization
        max_measures = max(self.M.values()) if self.M and len(self.M) > 0 else 1
        max_dim_coverage = 0.0
        max_diversity = 0.0

        # First pass: collect all values for proper normalization
        all_dim_coverages = []
        all_diversities = []
        all_measure_counts = []

        for mode, constructs in self.RELIABILITY_SENSITIVITY_MAPPING.items():
            mask = self.rai_df["canonical_principles"].apply(
                lambda x: any(p in x for p in constructs)
            )
            principle_measures = self.rai_df.loc[mask].copy()

            measure_count = len(principle_measures)
            dimension_coverage = self.calculate_dimension_coverage(
                principle_measures,
                self.rai_df
            )
            diversity = self.calculate_dimension_diversity(principle_measures)

            all_measure_counts.append(measure_count)
            all_dim_coverages.append(dimension_coverage)
            all_diversities.append(diversity)

            max_dim_coverage = max(max_dim_coverage, dimension_coverage)
            max_diversity = max(max_diversity, diversity)

        # If all values are zero, set max to 1 to avoid division by zero
        max_dim_coverage = max(max_dim_coverage, 0.01)
        max_diversity = max(max_diversity, 0.01)
        max_measures = max(max_measures, 1)

        # Second pass: compute normalized values
        for mode, constructs in self.RELIABILITY_SENSITIVITY_MAPPING.items():
            # Get measures for this mapping
            mask = self.rai_df["canonical_principles"].apply(
                lambda x: any(p in x for p in constructs)
            )
            principle_measures = self.rai_df.loc[mask].copy()

            measure_count = len(principle_measures)
            dimension_coverage = self.calculate_dimension_coverage(
                principle_measures,
                self.rai_df
            )
            diversity = self.calculate_dimension_diversity(principle_measures)

            # ============================================================
            # FIXED: Proper normalization for single values
            # ============================================================

            # 1. Measure Coverage: log-normalized against max
            if measure_count > 0:
                measure_coverage_norm = min(
                    np.log1p(measure_count) / np.log1p(max_measures),
                    1.0
                )
            else:
                measure_coverage_norm = 0.0

            # 2. Dimension Coverage: normalized against max
            if max_dim_coverage > 0:
                dimension_coverage_norm = min(
                    dimension_coverage / max_dim_coverage,
                    1.0
                )
            else:
                dimension_coverage_norm = 0.0

            # 3. Diversity: normalized against max
            if max_diversity > 0:
                diversity_norm = min(
                    diversity / max_diversity,
                    1.0
                )
            else:
                diversity_norm = 0.0

            # OI (Equal weight - 1/3 each)
            oi = (
                measure_coverage_norm +
                dimension_coverage_norm +
                diversity_norm
            ) / 3

            # EOGI
            eogi = p_norm - oi

            # Find matched constructs actually present
            matched_constructs = []
            for const in constructs:
                if const in self.M and self.M.get(const, 0) > 0:
                    matched_constructs.append(const)

            results.append({
                "mapping_mode": mode,
                "matched_constructs": (
                    ", ".join(matched_constructs)
                    if matched_constructs
                    else "none"
                ),
                "n_measures": measure_count,
                "dimension_coverage": dimension_coverage,
                "diversity": diversity,
                "measure_coverage_norm": measure_coverage_norm,
                "dimension_coverage_norm": dimension_coverage_norm,
                "diversity_norm": diversity_norm,
                "p_norm": p_norm,
                "oi": oi,
                "eogi": eogi
            })

        return pd.DataFrame(results)
    # ============================================================
    # RELIABILITY GAP
    # ============================================================

    def identify_reliability_gap(self) -> Dict:
        """Identify operationalization gap for the RELIABILITY principle."""
        reliability = "RELIABILITY"

        if reliability not in self.principles:
            return {
                "available": False,
                "reason": "Reliability principle not found."
            }

        prevalence = self.df[reliability].mean()
        median_prevalence = np.median([self.df[p].mean() for p in self.principles])
        measures = self.M.get(reliability, 0) if self.M is not None else None

        eogi_v2, oi = np.nan, np.nan
        if self.eogi_v2 is not None:
            row = self.eogi_v2[self.eogi_v2["principle"] == reliability]
            if not row.empty:
                eogi_v2 = float(row["eogi_v2"].iloc[0])
                oi = float(row["operationalization_index"].iloc[0])

        return {
            "available": True,
            "reliability_prevalence": float(prevalence),
            "reliability_measures": measures,
            "median_prevalence": float(median_prevalence),
            "operationalization_index": oi,
            "eogi_v2": eogi_v2,
            "high_normative_prominence": bool(prevalence > median_prevalence),
            "zero_rai_measures": measures == 0 if measures is not None else None,
            "operationalization_gap_pattern": (
                bool(prevalence > median_prevalence and measures == 0)
                if measures is not None else False
            )
        }

    # ============================================================
    # ROBUSTNESS SUMMARY
    # ============================================================

    def robustness_summary(self) -> Dict:
        """Summarize all robustness checks."""
        results = {}

        if self.clusters is not None:
            st = self.clusters["stability"]
            results["cluster_mean_ARI"] = st["mean_ari"]
            results["cluster_CI_lower"] = st["ci_lower"]
            results["cluster_CI_upper"] = st["ci_upper"]
            results["cluster_stable"] = st["stable"]

        if self.cluster_jaccard is not None:
            results["ari_hierarchical_vs_kmeans"] = self.cluster_jaccard["ari_with_kmeans_hierarchical"]
            results["ari_hamming_vs_kmeans"] = self.cluster_jaccard["ari_with_kmeans_hamming"]

        if self.eogi_baseline is not None and self.eogi_v2 is not None:
            merged = self.eogi_baseline[["principle", "eogi_baseline"]].merge(
                self.eogi_v2[["principle", "eogi_v2"]],
                on="principle"
            )
            rho, p = spearmanr(merged["eogi_baseline"], merged["eogi_v2"])
            results["baseline_v2_spearman_rho"] = float(rho)
            results["baseline_v2_spearman_p"] = float(p)

        if self.eogi_permutation is not None:
            results["eogi_permutation_p"] = self.eogi_permutation["p_value"]
            results["eogi_permutation_significant"] = self.eogi_permutation["significant"]

        if self.eogi_loo is not None:
            results["loo_mean_correlation"] = self.eogi_loo["mean_loo_correlation"]
            results["loo_min_correlation"] = self.eogi_loo["min_loo_correlation"]
            results["loo_stable"] = self.eogi_loo["stable"]

        if self.eogi_validation is not None:
            results["validation_r2_measure_only"] = self.eogi_validation["discriminant_validity"]["r2_measure_only"]

        return results

    # ============================================================
    # EXPORT RESULTS
    # ============================================================

    def export_results(self, output_dir: str) -> None:
        """Export all results to CSV files and JSON summary."""
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Main tables
        self.P.to_csv(output_dir / "principle_prevalence.csv", index=False)
        self.regional_analysis.to_csv(output_dir / "regional_analysis.csv", index=False)

        # Temporal trends
        temp_records = []
        for p, result in self.temporal.items():
            row = {"principle": p}
            row.update({k: v for k, v in result.items() if k != "yearly_data"})
            temp_records.append(row)
            result["yearly_data"].to_csv(output_dir / f"temporal_{p}.csv", index=False)
        pd.DataFrame(temp_records).to_csv(output_dir / "temporal_trends.csv", index=False)

        # Sen's slope
        sen_records = []
        for p, result in self.temporal_sen.items():
            row = {"principle": p}
            row.update({k: v for k, v in result.items() if k != "yearly_data"})
            sen_records.append(row)
        pd.DataFrame(sen_records).to_csv(output_dir / "sen_slope_trends.csv", index=False)

        # Regional statistics
        reg_summary = []
        for p, result in self.regional_statistics.items():
            reg_summary.append({
                "principle": p,
                "chi2": result["chi2"],
                "p_value": result["p_value"],
                "p_adjusted": result["p_adjusted"],
                "cramer_v": result["cramer_v"],
                "significant_fdr": result["significant_fdr"],
                "n": result["n"]
            })
            result["contingency"].to_csv(
                output_dir / f"regional_contingency_{p}.csv"
            )
        pd.DataFrame(reg_summary).to_csv(
            output_dir / "regional_statistics.csv", index=False
        )

        # Network
        if self.network:
            pd.DataFrame(self.network["all_relationships"]).to_csv(
                output_dir / "network_relationships.csv", index=False
            )
            pd.DataFrame(self.network["validated_relationships"]).to_csv(
                output_dir / "network_validated_relationships.csv", index=False
            )
            self.network["centrality"].to_csv(
                output_dir / "network_centrality.csv"
            )
            self.network["phi_matrix"].to_csv(
                output_dir / "phi_matrix.csv"
            )

        # Clustering
        if self.clusters:
            self.clusters["profiles"].to_csv(
                output_dir / "cluster_profiles.csv"
            )
            self.clusters["regional"].to_csv(
                output_dir / "cluster_regional_distribution.csv"
            )
            self.clusters["institutional"].to_csv(
                output_dir / "cluster_institutional_distribution.csv"
            )
            self.clusters["selection"]["scores"].to_csv(
                output_dir / "cluster_selection.csv", index=False
            )
            pd.DataFrame({
                "metric": ["mean_ari", "ci_lower", "ci_upper", "stable"],
                "value": [
                    self.clusters["stability"]["mean_ari"],
                    self.clusters["stability"]["ci_lower"],
                    self.clusters["stability"]["ci_upper"],
                    self.clusters["stability"]["stable"]
                ]
            }).to_csv(output_dir / "cluster_stability.csv", index=False)

        # Jaccard clustering
        if self.cluster_jaccard:
            pd.DataFrame({
                "metric": [
                    "ari_hierarchical_vs_kmeans",
                    "ari_hamming_vs_kmeans",
                    "n_clusters"
                ],
                "value": [
                    self.cluster_jaccard["ari_with_kmeans_hierarchical"],
                    self.cluster_jaccard["ari_with_kmeans_hamming"],
                    self.cluster_jaccard["n_clusters"]
                ]
            }).to_csv(output_dir / "cluster_jaccard_robustness.csv", index=False)

        # RAI
        if self.rai_profile is not None:
            self.rai_profile.to_csv(
                output_dir / "rai_operationalization_profile.csv", index=False
            )

        if self.operationalization_index is not None:
            self.operationalization_index.to_csv(
                output_dir / "operationalization_index.csv", index=False
            )

        # EOGI
        if self.eogi_baseline is not None:
            self.eogi_baseline.to_csv(
                output_dir / "eogi_baseline.csv", index=False
            )

        if self.eogi_v2 is not None:
            self.eogi_v2.to_csv(
                output_dir / "eogi_v2.csv", index=False
            )

        if self.eogi_bootstrap is not None:
            self.eogi_bootstrap.to_csv(
                output_dir / "eogi_bootstrap.csv", index=False
            )

        if self.weight_sensitivity is not None:
            self.weight_sensitivity.to_csv(
                output_dir / "eogi_weight_sensitivity.csv", index=False
            )

        if self.reliability_sensitivity is not None:
            self.reliability_sensitivity.to_csv(
                output_dir / "reliability_sensitivity.csv", index=False
            )

        if self.eogi_validation is not None:
            with open(output_dir / "eogi_validation.json", "w", encoding="utf-8") as f:
                json.dump(self.eogi_validation, f, indent=4, default=str)

        # RAI diagnostics
        if self.rai_df is not None:
            rai_diagnostics = {
                "total_rows": len(self.rai_raw) if self.rai_raw is not None else 0,
                "valid_rows": len(self.rai_df),
                "unique_measures": self.rai_df["measure_id"].nunique(),
                "total_canonical_entries": sum(self.M.values()) if self.M else 0,
                "unique_principles": len(self.M) if self.M else 0
            }
            # Add dimension column diagnostics
            for col in self.RAI_DIMENSION_COLUMNS:
                if col in self.rai_df.columns:
                    rai_diagnostics[f"{col}_non_null"] = self.rai_df[col].notna().sum()
                    rai_diagnostics[f"{col}_unique"] = self.rai_df[col].nunique()

            pd.DataFrame([rai_diagnostics]).to_csv(
                output_dir / "rai_diagnostics.csv", index=False
            )

        # Summary
        summary = {
            "framework": "Ethical Operationalization Framework v5.1",
            "random_seed": self.seed,
            "n_documents": int(len(self.df)),
            "n_countries": int(self.df["country"].nunique()),
            "n_regions": int(self.df["region"].nunique()),
            "year_min": float(self.df["year"].min()) if self.df["year"].notna().any() else None,
            "year_max": float(self.df["year"].max()) if self.df["year"].notna().any() else None,
            "n_principles": len(self.principles),
            "principles": self.principles,
            "n_rai_measures": int(len(self.rai_df)) if self.rai_df is not None else 0,
            "cluster_k": self.clusters["n_clusters"] if self.clusters else None,
            "network_density": self.network["network_density"] if self.network else None,
            "n_validated_network_edges": len(self.network["validated_relationships"]) if self.network else 0,
            "robustness": self.robustness_summary()
        }

        with open(output_dir / "analysis_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=4, ensure_ascii=False, default=str)

        logger.info(f"Results exported to: {output_dir}")


# ================================================================
# FIGURE GENERATION
# ================================================================

def save_html(fig, output_dir, filename, show=True):
    """Save figure as HTML with CDN Plotly and optionally display it."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(output_dir / filename), include_plotlyjs="cdn")

    # عرض الشكل على الشاشة
    if show:
        fig.show()

    return fig

def plot_figure_1(analyzer, output_dir, show=True):
    """Figure 1: Principle Prevalence with Wilson CI."""
    df = analyzer.P.copy()
    df["error_lower"] = df["prevalence"] - df["ci_low"]
    df["error_upper"] = df["ci_high"] - df["prevalence"]

    fig = px.bar(
        df,
        x="prevalence",
        y="principle",
        orientation="h",
        title="Figure 1: Global Prevalence of AI Ethics Principles (95% Wilson CI)",
        labels={"prevalence": "Prevalence", "principle": "Ethical Principle"},
        error_x="error_upper",
        error_x_minus="error_lower",
        height=700
    )
    fig.update_layout(margin=dict(l=0, r=0, t=70, b=20))
    return save_html(fig, output_dir, "figure_1_prevalence.html", show=show)


def plot_figure_2(analyzer, output_dir, show=True):
    """Figure 2: Temporal Trends."""
    top_principles = analyzer.P.head(5)["principle"].tolist()
    temporal_df = analyzer.df.dropna(subset=["year"])
    if temporal_df.empty:
        return None

    yearly = temporal_df.groupby("year")[top_principles].mean().reset_index()

    fig = px.line(
        yearly,
        x="year",
        y=top_principles,
        title="Figure 2: Temporal Trends in Key Ethical Principles",
        labels={"value": "Prevalence", "year": "Year"},
        height=550
    )
    return save_html(fig, output_dir, "figure_2_temporal_trends.html", show=show)


def plot_figure_3(analyzer, output_dir, show=True):
    """Figure 3: Regional Heatmap."""
    pivot = analyzer.regional_analysis.pivot_table(
        index="principle", columns="region", values="prevalence"
    )
    pivot = pivot.reindex(analyzer.P["principle"])

    fig = px.imshow(
        pivot,
        title="Figure 3: Ethical Principle Prevalence by Region",
        aspect="auto",
        labels={"x": "Region", "y": "Principle", "color": "Prevalence"},
        height=700
    )
    return save_html(fig, output_dir, "figure_3_regional_heatmap.html", show=show)


def plot_figure_4(analyzer, output_dir, show=True):
    """Figure 4: Cluster Profiles."""
    if analyzer.clusters is None:
        return None

    fig = px.imshow(
        analyzer.clusters["profiles"].T,
        title="Figure 4: Cluster Profiles of Ethical Principles",
        aspect="auto",
        labels={"x": "Cluster", "y": "Principle", "color": "Prevalence"},
        height=700
    )
    return save_html(fig, output_dir, "figure_4_cluster_profiles.html", show=show)


def plot_figure_5(analyzer, output_dir, show=True):
    """Figure 5: EOGI Comparison."""
    if analyzer.eogi_baseline is None or analyzer.eogi_v2 is None:
        return None

    merged = analyzer.eogi_baseline[["principle", "eogi_baseline"]].merge(
        analyzer.eogi_v2[["principle", "eogi_v2"]],
        on="principle"
    )

    fig = px.scatter(
        merged,
        x="eogi_baseline",
        y="eogi_v2",
        text="principle",
        title="Figure 5: Baseline EOGI vs Multidimensional EOGI v2",
        labels={
            "eogi_baseline": "Baseline EOGI (Count-based)",
            "eogi_v2": "Multidimensional EOGI v2"
        },
        height=600
    )

    min_val = min(merged["eogi_baseline"].min(), merged["eogi_v2"].min())
    max_val = max(merged["eogi_baseline"].max(), merged["eogi_v2"].max())

    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(dash="dash"),
        name="Identity"
    ))

    fig.update_traces(textposition="top center")
    return save_html(fig, output_dir, "figure_5_eogi_comparison.html", show=show)


def plot_figure_6(analyzer, output_dir, show=True):
    """Figure 6: Normative Prominence vs Operationalization."""
    if analyzer.eogi_v2 is None:
        return None

    df = analyzer.eogi_v2.copy()

    fig = px.scatter(
        df,
        x="prevalence_norm",
        y="operationalization_index",
        text="principle",
        size="measure_count",
        color="gap_category",
        title="Figure 6: Normative Prominence vs RAI Operationalization",
        labels={
            "prevalence_norm": "Normalized Guideline Prevalence (P)",
            "operationalization_index": "RAI Operationalization Index (OI)"
        },
        height=650
    )

    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Alignment (P=OI)"
    ))

    fig.update_traces(textposition="top center")
    return save_html(fig, output_dir, "figure_6_normative_vs_operationalization.html", show=show)


def plot_figure_7(analyzer, output_dir, show=True):
    """Figure 7: Phi Network."""
    if analyzer.network is None:
        return None

    fig = px.imshow(
        analyzer.network["phi_matrix"],
        title="Figure 7: Ethical Principle Co-occurrence (Signed Phi)",
        zmin=-1,
        zmax=1,
        aspect="auto",
        labels={"x": "Principle", "y": "Principle", "color": "Phi"},
        height=750
    )
    return save_html(fig, output_dir, "figure_7_phi_network.html", show=show)


def plot_figure_8(analyzer, output_dir, show=True):
    """Figure 8: EOGI Gap."""
    if analyzer.eogi_v2 is None:
        return None

    df = analyzer.eogi_v2.copy()

    fig = px.bar(
        df.sort_values("eogi_v2"),
        x="eogi_v2",
        y="principle",
        orientation="h",
        color="gap_category",
        title="Figure 8: Multidimensional Operationalization Gap (EOGI v2)",
        labels={
            "eogi_v2": "EOGI v2 (P - OI)",
            "principle": "Ethical Principle"
        },
        height=700
    )

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_vline(x=0.25, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=0.50, line_dash="dash", line_color="gray", opacity=0.5)

    return save_html(fig, output_dir, "figure_8_eogi_gap.html", show=show)


def plot_figure_9(analyzer, output_dir, show=True):
    """Figure 9: Reliability Sensitivity Analysis."""
    if analyzer.reliability_sensitivity is None:
        return None

    df = analyzer.reliability_sensitivity.copy()

    fig = px.bar(
        df,
        x="mapping_mode",
        y="eogi",
        title="Figure 9: Reliability EOGI Sensitivity to Mapping Definition",
        labels={
            "mapping_mode": "Reliability Mapping",
            "eogi": "EOGI v2"
        },
        height=500,
        text_auto=True
    )

    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.update_traces(textposition="outside")

    return save_html(fig, output_dir, "figure_9_reliability_sensitivity.html", show=show)
# ================================================================
# PRINT RESULTS
# ================================================================

def print_results(analyzer):
    """Print comprehensive results to console."""
    print("\n" + "=" * 115)
    print("ETHICAL OPERATIONALIZATION FRAMEWORK v5.1")
    print("Publication-Ready Version - RAI Fully Restructured")
    print("=" * 115)

    # Sample Characteristics
    print("\n[1] SAMPLE CHARACTERISTICS")
    print("-" * 80)
    print(f"Total Documents: {len(analyzer.df)}")
    print(f"Countries: {analyzer.df['country'].nunique()}")
    print(f"Regions: {analyzer.df['region'].nunique()}")
    if analyzer.df["year"].notna().any():
        print(f"Years: {int(analyzer.df['year'].min())} - {int(analyzer.df['year'].max())}")
    print(f"Principles Analyzed: {len(analyzer.principles)}")
    if analyzer.rai_df is not None:
        print(f"RAI Measures (valid): {len(analyzer.rai_df)}")
        print(f"RAI Columns: {list(analyzer.rai_df.columns)[:10]}...")

    # Prevalence
    print("\n[2] PRINCIPLE PREVALENCE (Wilson 95% CI)")
    print("-" * 105)
    print(f"{'Principle':<35}{'Prevalence':<15}{'95% CI':<25}{'Documents':<12}")
    print("-" * 105)
    for _, row in analyzer.P.iterrows():
        print(f"{row['principle']:<35}{row['prevalence']:<15.2%}({row['ci_low']:.3f}, {row['ci_high']:.3f}){row['documents']:<12}")

    # Temporal
    print("\n[3] TEMPORAL TRENDS (Sen's Slope)")
    print("-" * 110)
    print(f"{'Principle':<35}{'Sen Slope':<15}{'Kendall τ':<15}{'p-value':<12}")
    print("-" * 110)
    for p, result in sorted(
        analyzer.temporal_sen.items(),
        key=lambda x: x[1]["slope"] if np.isfinite(x[1]["slope"]) else -np.inf,
        reverse=True
    )[:10]:
        slope = result["slope"] if np.isfinite(result["slope"]) else 0
        tau = result["kendall_tau"] if np.isfinite(result["kendall_tau"]) else 0
        p_val = result["p_value"] if np.isfinite(result["p_value"]) else 1
        print(f"{p:<35}{slope:>+12.5f}{'':3}{tau:<15.3f}{p_val:<12.4f}")

    # Regional
    print("\n[4] REGIONAL DIFFERENCES (χ² + Cramér's V)")
    print("-" * 110)
    print(f"{'Principle':<35}{'Chi2':<12}{'p_adj':<14}{'Cramer V':<14}{'FDR':<12}")
    print("-" * 110)
    for p, result in analyzer.regional_statistics.items():
        print(f"{p:<35}{result['chi2']:<12.3f}{result['p_adjusted']:<14.5f}{result['cramer_v']:<14.3f}{'SIGNIFICANT' if result['significant_fdr'] else 'NS':<12}")

    # Network
    print("\n[5] NETWORK ANALYSIS")
    print("-" * 80)
    print(f"Network Density: {analyzer.network['network_density']:.3%}")
    print(f"Validated Relationships: {len(analyzer.network['validated_relationships'])}")
    print("\nTop Central Principles:")
    for p, row in analyzer.network["centrality"].head(10).iterrows():
        print(f"  {p:<35}Degree={int(row['degree']):<5}Weighted Degree={row['weighted_degree']:.3f}")

    # Clustering
    print("\n[6] CLUSTER ANALYSIS")
    print("-" * 80)
    clusters = analyzer.clusters
    print(f"Selected K: {clusters['n_clusters']}")
    print(f"Bootstrap ARI: {clusters['stability']['mean_ari']:.3f} ± {clusters['stability']['std_ari']:.3f}")
    print(f"95% CI: [{clusters['stability']['ci_lower']:.3f}, {clusters['stability']['ci_upper']:.3f}]")
    print(f"Stable: {clusters['stability']['stable']}")
    if analyzer.cluster_jaccard:
        print(f"Jaccard/Hamming Robustness ARI: {analyzer.cluster_jaccard['ari_with_kmeans_hierarchical']:.3f}")
    print("\nCluster Sizes:")
    for cid, size in analyzer.df["cluster"].value_counts().sort_index().items():
        print(f"  Cluster {cid}: {size} documents ({size/len(analyzer.df):.1%})")

    # RAI
    if analyzer.rai_df is not None:
        print("\n[7] RAI OPERATIONALIZATION")
        print("-" * 115)
        oi = analyzer.operationalization_index
        print(f"{'Principle':<35}{'Measures':<10}{'Coverage':<13}{'Dimensions':<14}{'Diversity':<13}{'OI':<10}")
        print("-" * 115)
        for _, row in oi.iterrows():
            print(f"{row['principle']:<35}{int(row['measure_count']):<10}{row['measure_coverage_norm']:<13.3f}{row['dimension_coverage_norm']:<14.3f}{row['diversity_norm']:<13.3f}{row['operationalization_index']:<10.3f}")

        # EOGI
        print("\n[8] ETHICAL OPERATIONALIZATION GAP INDEX (EOGI v2)")
        print("-" * 110)
        print(f"{'Principle':<35}{'P_norm':<12}{'OI':<12}{'EOGI':<12}{'Category':<40}")
        print("-" * 110)
        for _, row in analyzer.eogi_v2.iterrows():
            print(f"{row['principle']:<35}{row['prevalence_norm']:<12.3f}{row['operationalization_index']:<12.3f}{row['eogi_v2']:+.3f}     {row['gap_category']:<40}")

        # Reliability Sensitivity
        if analyzer.reliability_sensitivity is not None:
            print("\n[9] RELIABILITY SENSITIVITY ANALYSIS")
            print("-" * 115)
            print(f"{'Mapping Mode':<20}{'Constructs':<45}{'Measures':<10}{'OI':<12}{'EOGI':<12}")
            print("-" * 115)
            for _, row in analyzer.reliability_sensitivity.iterrows():
                constructs = row['matched_constructs'][:42] + "..." if len(str(row['matched_constructs'])) > 42 else row['matched_constructs']
                print(f"{row['mapping_mode']:<20}{constructs:<45}{int(row['n_measures']):<10}{row['oi']:<12.3f}{row['eogi']:<12.3f}")

        # Validation
        print("\n[10] EOGI VALIDATION")
        print("-" * 80)
        if analyzer.eogi_validation is not None:
            val = analyzer.eogi_validation
            print("Construct Validity (Spearman correlations):")
            for k, v in val["construct_validity"].items():
                status = v.get("status", "computed")
                print(f"  {k}: r = {v['correlation']:.3f}, p = {v['p_value']:.4f} [{status}]")
            print(f"Discriminant Validity (R² with measure count): {val['discriminant_validity']['r2_measure_only']:.3f}")

        # Robustness
        print("\n[11] ROBUSTNESS SUMMARY")
        print("-" * 80)
        robust = analyzer.robustness_summary()
        for key, value in robust.items():
            if isinstance(value, float):
                print(f"{key}: {value:.4f}")
            else:
                print(f"{key}: {value}")

        # Reliability Gap
        print("\n[12] RELIABILITY OPERATIONALIZATION GAP")
        print("-" * 80)
        par = analyzer.paradox
        if par["available"]:
            print(f"Reliability prevalence: {par['reliability_prevalence']:.2%}")
            print(f"Reliability RAI measures: {par['reliability_measures']}")
            print(f"Reliability OI: {par['operationalization_index']:.3f}")
            print(f"Reliability EOGI v2: {par['eogi_v2']:+.3f}")
            print(f"High normative prominence: {par['high_normative_prominence']}")
            print(f"Zero RAI measures: {par['zero_rai_measures']}")
            print(f"Operationalization gap pattern: {par['operationalization_gap_pattern']}")

    print("\n" + "=" * 115)
    print("ANALYSIS COMPLETE")
    print("=" * 115)


# ================================================================
# MAIN
# ================================================================
def main():
    """Main execution function."""
    guidelines_path = "/kaggle/working/worldwide_AI-ethicss/data"
    rai_path = "/kaggle/input/datasets/drabdulbasetaledresi/ai-ethics/Version_1.0_RAI_Measures_Dataset.csv"
    output_dir = "/kaggle/working/ethical_operationalization_results_v5"

    print("=" * 115)
    print("INITIALIZING ETHICAL OPERATIONALIZATION FRAMEWORK v5.1")
    print("Publication-Ready Version - RAI Fully Restructured")
    print("=" * 115)

    # Validate paths
    if not os.path.exists(guidelines_path):
        raise FileNotFoundError(f"Guidelines path not found:\n{guidelines_path}")

    if not os.path.exists(rai_path):
        logger.warning("RAI dataset not found. Continuing without RAI analysis.")
        rai_path = None

    # Initialize
    analyzer = EthicalOperationalizationFramework(
        guidelines_path=guidelines_path,
        rai_data_path=rai_path,
        seed=42,
        n_permutations=5000,
        n_bootstrap=1000
    )

    # Print results
    print_results(analyzer)

    # Run bootstrap (if RAI available)
    if analyzer.rai_df is not None:
        print("\n" + "=" * 115)
        print("RUNNING EOGI BOOTSTRAP")
        print("=" * 115)

        eogi_bootstrap = analyzer.bootstrap_eogi_v2(n_bootstrap=1000)
        analyzer.eogi_bootstrap = eogi_bootstrap

        if eogi_bootstrap is not None:
            print("\nEOGI v2 Bootstrap 95% Confidence Intervals:")
            print("-" * 100)
            for _, row in eogi_bootstrap.iterrows():
                print(f"{row['principle']:<35}Mean={row['eogi_mean']:+.3f} | CI=[{row['ci_lower']:+.3f}, {row['ci_upper']:+.3f}]")

            os.makedirs(output_dir, exist_ok=True)
            eogi_bootstrap.to_csv(
                os.path.join(output_dir, "eogi_bootstrap.csv"),
                index=False
            )

    # Export results
    print("\n" + "=" * 115)
    print("EXPORTING RESULTS")
    print("=" * 115)
    analyzer.export_results(output_dir)

    # Generate AND DISPLAY figures
    print("\n" + "=" * 115)
    print("GENERATING AND DISPLAYING FIGURES")
    print("=" * 115)

    # عرض الأشكال على الشاشة مع حفظها
    plot_figure_1(analyzer, output_dir, show=True)
    plot_figure_2(analyzer, output_dir, show=True)
    plot_figure_3(analyzer, output_dir, show=True)
    plot_figure_4(analyzer, output_dir, show=True)

    if analyzer.rai_df is not None:
        plot_figure_5(analyzer, output_dir, show=True)
        plot_figure_6(analyzer, output_dir, show=True)
        plot_figure_8(analyzer, output_dir, show=True)
        plot_figure_9(analyzer, output_dir, show=True)

    plot_figure_7(analyzer, output_dir, show=True)

    # Final summary
    print("\n" + "=" * 115)
    print("ALL ANALYSES AND FIGURES COMPLETED")
    print(f"\nResults saved to:\n{output_dir}")
    print("\nKey Output Files:")
    print(f"  - principle_prevalence.csv (Principle prevalence with CI)")
    print(f"  - eogi_v2.csv (EOGI v2 results)")
    print(f"  - reliability_sensitivity.csv (Reliability sensitivity analysis)")
    print(f"  - eogi_validation.json (EOGI validation metrics)")
    print(f"  - rai_diagnostics.csv (RAI dataset diagnostics)")
    print(f"  - analysis_summary.json (Complete summary)")
    print(f"  - figure_*.html (9 publication-ready figures)")
    print("=" * 115)

    return analyzer


if __name__ == "__main__":
    analyzer = main()


INITIALIZING ETHICAL OPERATIONALIZATION FRAMEWORK v5.1
Publication-Ready Version - RAI Fully Restructured

ETHICAL OPERATIONALIZATION FRAMEWORK v5.1
Publication-Ready Version - RAI Fully Restructured

[1] SAMPLE CHARACTERISTICS
--------------------------------------------------------------------------------
Total Documents: 200
Countries: 46
Regions: 23
Years: 2014 - 2022
Principles Analyzed: 15
RAI Measures (valid): 791
RAI Columns: ['Target Output', 'Unnamed: 1', 'Entry Points', 'Unnamed: 3', 'Connections to Harm', 'Unnamed: 5', 'Measurement Properties', 'Unnamed: 7', 'Unnamed: 8', 'Algorithmic System Characteristics']...

[2] PRINCIPLE PREVALENCE (Wilson 95% CI)
---------------------------------------------------------------------------------------------------------
Principle                          Prevalence     95% CI                   Documents   
---------------------------------------------------------------------------------------------------------
TRANSPARENCY              


ALL ANALYSES AND FIGURES COMPLETED

Results saved to:
/kaggle/working/ethical_operationalization_results_v5

Key Output Files:
  - principle_prevalence.csv (Principle prevalence with CI)
  - eogi_v2.csv (EOGI v2 results)
  - reliability_sensitivity.csv (Reliability sensitivity analysis)
  - eogi_validation.json (EOGI validation metrics)
  - rai_diagnostics.csv (RAI dataset diagnostics)
  - analysis_summary.json (Complete summary)
  - figure_*.html (9 publication-ready figures)
